# SLAP2 acquisition TIFF cleanup

Safely remove large **per-cycle acquisition TIFFs** while retaining the corresponding raw `.dat` files and all reference/static TIFFs.

**Safety defaults**
- targets only `slap2/dynamic_data/acquisition_*CYCLE-*.tif`
- uses the manifest to identify mouse/session membership
- requires the matching `.dat` to be both **manifested and present on disk**
- starts in **dry-run mode**
- never targets files containing `REFERENCE`, `refStack`, or `structure_volume`
- writes a CSV plan/log before deletion
- optional per-session confirmation when deleting

Edit only the configuration cell first.

In [1]:
from pathlib import Path, PureWindowsPath
from datetime import datetime
import pandas as pd
import re
import csv

# =========================
# CONFIGURATION
# =========================

# Directory against which manifest paths are relative.
DATA_ROOT = Path(r"\\allen\aind\scratch\ophys\Andrew\VIP_synaptic_dynamics")

# One or more manifests may be combined.
# Example:
# MANIFEST_PATHS = [
#     DATA_ROOT / "directory_manifest.csv",
#     Path(r"C:\another\directory_manifest.csv"),
# ]
MANIFEST_PATHS = [
    DATA_ROOT / "directory_manifest.tsv",
]

# Selection:
#   None -> no filtering on that field.
#   A list -> only those mice/sessions.
SELECT_MICE = None
# SELECT_MICE = ["826031", "826032","852835","863774"
#               '803496','804730','804733','810196','809047','803121',
#               '826033','838410','834788','852834']

SELECT_SESSIONS = None
# SELECT_SESSIONS = ["852835_2026-07-27_11-59-54"]

# Keep this True unless you intentionally want to consider non-CYCLE
# acquisition TIFFs too.
CYCLE_ONLY = True

# Safety checks.
REQUIRE_DAT_IN_MANIFEST = True
REQUIRE_DAT_ON_DISK = True

# Deletion controls.
DRY_RUN = False
PROMPT_EACH_SESSION = False

# Deletion only proceeds when DRY_RUN=False.
# With PROMPT_EACH_SESSION=True, each session additionally requires
# typing: DELETE <session_id>
GLOBAL_CONFIRMATION = "DELETE SLAP2 ACQUISITION TIFFS"
REQUIRED_GLOBAL_CONFIRMATION = "DELETE SLAP2 ACQUISITION TIFFS"

LOG_DIR = DATA_ROOT / "_slap2_cleanup_logs"


## 1. Load and parse manifest(s)

The attached manifest is tab-separated despite its `.csv` extension, so this loader auto-detects tab/comma delimiters and accepts either headerless `path + type` manifests or simple named-column manifests.

In [2]:
SESSION_RE = re.compile(
    r"(?P<session>(?P<mouse>\d+)_(?P<date>\d{4}-\d{2}-\d{2})_(?P<time>\d{2}-\d{2}-\d{2}))"
)

def _detect_delimiter(path: Path) -> str:
    sample = path.read_text(errors="replace")[:8192]
    try:
        return csv.Sniffer().sniff(sample, delimiters="\t,;").delimiter
    except csv.Error:
        return "\t" if "\t" in sample else ","

def load_manifest(path: Path) -> pd.DataFrame:
    path = Path(path)
    if not path.exists():
        raise FileNotFoundError(f"Manifest not found: {path}")

    sep = _detect_delimiter(path)

    # First try headerless path/type, which matches the supplied manifest.
    raw = pd.read_csv(
        path, sep=sep, header=None, names=["relative_path", "kind"],
        dtype=str, keep_default_na=False
    )

    # If the first row looks like a real header, reload using it.
    first_path = str(raw.iloc[0]["relative_path"]).strip().lower()
    first_kind = str(raw.iloc[0]["kind"]).strip().lower()
    if first_path in {"path", "relative_path", "filepath", "file_path"} or first_kind in {"type", "kind"}:
        named = pd.read_csv(path, sep=sep, dtype=str, keep_default_na=False)
        lookup = {c.lower(): c for c in named.columns}
        path_col = next((lookup[k] for k in ["relative_path", "path", "filepath", "file_path"] if k in lookup), None)
        kind_col = next((lookup[k] for k in ["kind", "type", "entry_type"] if k in lookup), None)
        if path_col is None:
            raise ValueError(f"Could not identify a path column in {path}")
        raw = pd.DataFrame({
            "relative_path": named[path_col].astype(str),
            "kind": named[kind_col].astype(str) if kind_col else "file",
        })

    raw["manifest"] = str(path)
    return raw

def parse_session_fields(rel_path: str):
    m = SESSION_RE.search(str(rel_path))
    if not m:
        return pd.Series({"mouse": None, "session": None})
    return pd.Series({"mouse": m.group("mouse"), "session": m.group("session")})

manifest = pd.concat([load_manifest(p) for p in MANIFEST_PATHS], ignore_index=True)
manifest[["mouse", "session"]] = manifest["relative_path"].apply(parse_session_fields)

sessions = (
    manifest.dropna(subset=["mouse", "session"])[["mouse", "session"]]
    .drop_duplicates()
    .sort_values(["mouse", "session"])
    .reset_index(drop=True)
)

print(f"Loaded {len(manifest):,} manifest entries from {len(MANIFEST_PATHS)} manifest(s).")
print(f"Found {sessions['mouse'].nunique():,} mice and {len(sessions):,} sessions.")
display(sessions)


Loaded 94,457 manifest entries from 1 manifest(s).
Found 34 mice and 151 sessions.


,mouse,session
0,803121,803121_2025-10-06_17-44-40
1,803121,803121_2025-10-29_11-19-29
2,803121,803121_2025-10-29_11-32-00
3,803121,803121_2025-10-30_11-13-32
4,803121,803121_2025-10-31_13-05-26
...,...,...
146,863774,863774_2026-08-06_09-15-41
147,863774,863774_2026-08-07_13-29-18
148,863774,863774_2026-08-08_13-35-42
149,863774,863774_2026-08-09_11-37-01


## 2. Build the cleanup plan

A TIFF is **eligible** only if it passes the filename/path rules and its corresponding `.dat` passes the configured safety checks. This cell does not delete anything.

In [3]:
PROTECTED_TERMS = ("reference", "refstack", "structure_volume")

def windows_parts(rel_path: str):
    p = PureWindowsPath(str(rel_path))
    if p.is_absolute() or ".." in p.parts:
        raise ValueError(f"Unsafe manifest path: {rel_path}")
    return p.parts

def disk_path(rel_path: str) -> Path:
    return DATA_ROOT.joinpath(*windows_parts(rel_path))

def is_candidate_tiff(rel_path: str) -> bool:
    p = PureWindowsPath(str(rel_path))
    low_parts = [x.lower() for x in p.parts]
    low_name = p.name.lower()

    # Must be inside .../slap2/dynamic_data/...
    try:
        i = low_parts.index("slap2")
        in_dynamic_data = i + 1 < len(low_parts) and low_parts[i + 1] == "dynamic_data"
    except ValueError:
        in_dynamic_data = False

    if not in_dynamic_data:
        return False
    if p.suffix.lower() not in {".tif", ".tiff"}:
        return False
    if not low_name.startswith("acquisition_"):
        return False
    if CYCLE_ONLY and "-cycle-" not in low_name:
        return False
    if any(term in low_name for term in PROTECTED_TERMS):
        return False
    return True

def corresponding_dat_rel(rel_path: str) -> str:
    p = PureWindowsPath(str(rel_path))
    return str(p.with_suffix(".dat"))

def human_bytes(n):
    if pd.isna(n):
        return "NA"
    n = float(n)
    for unit in ["B", "KB", "MB", "GB", "TB"]:
        if n < 1024 or unit == "TB":
            return f"{n:.2f} {unit}"
        n /= 1024

manifest_file_paths_lower = set(
    manifest.loc[manifest["kind"].str.lower().eq("file"), "relative_path"]
    .astype(str).str.lower()
)

plan = manifest[
    manifest["kind"].str.lower().eq("file")
    & manifest["relative_path"].map(is_candidate_tiff)
].copy()

if SELECT_MICE is not None:
    wanted = {str(x) for x in SELECT_MICE}
    plan = plan[plan["mouse"].astype(str).isin(wanted)]

if SELECT_SESSIONS is not None:
    wanted = {str(x) for x in SELECT_SESSIONS}
    plan = plan[plan["session"].astype(str).isin(wanted)]

plan["tif_path"] = plan["relative_path"].map(disk_path)
plan["dat_relative_path"] = plan["relative_path"].map(corresponding_dat_rel)
plan["dat_path"] = plan["dat_relative_path"].map(disk_path)

plan["tif_exists"] = plan["tif_path"].map(Path.exists)
plan["dat_exists"] = plan["dat_path"].map(Path.exists)
plan["dat_manifested"] = plan["dat_relative_path"].str.lower().isin(manifest_file_paths_lower)

def safe_size(p):
    try:
        return p.stat().st_size
    except OSError:
        return pd.NA

plan["bytes"] = plan["tif_path"].map(safe_size)

plan["eligible"] = plan["tif_exists"]
if REQUIRE_DAT_ON_DISK:
    plan["eligible"] &= plan["dat_exists"]
if REQUIRE_DAT_IN_MANIFEST:
    plan["eligible"] &= plan["dat_manifested"]

def block_reason(row):
    reasons = []
    if not row["tif_exists"]:
        reasons.append("TIFF missing")
    if REQUIRE_DAT_ON_DISK and not row["dat_exists"]:
        reasons.append("matching DAT missing")
    if REQUIRE_DAT_IN_MANIFEST and not row["dat_manifested"]:
        reasons.append("matching DAT not in manifest")
    return "; ".join(reasons)

plan["block_reason"] = plan.apply(block_reason, axis=1)
plan = plan.sort_values(["mouse", "session", "relative_path"]).reset_index(drop=True)

summary = (
    plan.groupby(["mouse", "session"], dropna=False)
    .agg(
        candidates=("relative_path", "size"),
        eligible=("eligible", "sum"),
        existing_tiffs=("tif_exists", "sum"),
        bytes=("bytes", lambda x: pd.to_numeric(x, errors="coerce").sum(min_count=1)),
    )
    .reset_index()
)
summary["size"] = summary["bytes"].map(human_bytes)

display(summary.drop(columns="bytes"))
print()
print(f"Candidate TIFFs: {len(plan):,}")
print(f"Eligible TIFFs:  {int(plan['eligible'].sum()):,}")
print(f"Eligible size:   {human_bytes(pd.to_numeric(plan.loc[plan['eligible'], 'bytes'], errors='coerce').sum(min_count=1))}")

blocked = plan[~plan["eligible"]]
if len(blocked):
    print(f"\nBlocked by safety checks: {len(blocked):,}")
    display(blocked[["mouse", "session", "relative_path", "block_reason"]].head(50))


,mouse,session,candidates,eligible,existing_tiffs,size
0,803121,803121_2025-10-06_17-44-40,74,0,0,NA
1,814591,814591_2025-11-07_10-21-49,52,0,0,NA
2,814593,814593_2025-11-07_13-36-49,78,0,0,NA
3,824806,824806_2026-01-08_11-12-01,757,0,0,NA
4,825854,825854_2026-01-12_11-09-10,37,0,0,NA
...,...,...,...,...,...,...
67,863774,863774_2026-08-07_13-29-18,166,166,166,289.04 GB
68,863774,863774_2026-08-08_13-35-42,179,179,179,302.59 GB
69,863774,863774_2026-08-09_11-37-01,307,307,307,584.38 GB
70,875436,875436_2026-07-20_11-20-21,53,53,53,72.01 GB



Candidate TIFFs: 15,941
Eligible TIFFs:  10,429
Eligible size:   15.50 TB

Blocked by safety checks: 5,512


,mouse,session,relative_path,block_reason
0,803121,803121_2025-10-06_17-44-40,iGluSnFR4f\803121\2025-10-06_803121\803121_202...,TIFF missing
1,803121,803121_2025-10-06_17-44-40,iGluSnFR4f\803121\2025-10-06_803121\803121_202...,TIFF missing
2,803121,803121_2025-10-06_17-44-40,iGluSnFR4f\803121\2025-10-06_803121\803121_202...,TIFF missing
3,803121,803121_2025-10-06_17-44-40,iGluSnFR4f\803121\2025-10-06_803121\803121_202...,TIFF missing
4,803121,803121_2025-10-06_17-44-40,iGluSnFR4f\803121\2025-10-06_803121\803121_202...,TIFF missing
5,803121,803121_2025-10-06_17-44-40,iGluSnFR4f\803121\2025-10-06_803121\803121_202...,TIFF missing
6,803121,803121_2025-10-06_17-44-40,iGluSnFR4f\803121\2025-10-06_803121\803121_202...,TIFF missing
7,803121,803121_2025-10-06_17-44-40,iGluSnFR4f\803121\2025-10-06_803121\803121_202...,TIFF missing
8,803121,803121_2025-10-06_17-44-40,iGluSnFR4f\803121\2025-10-06_803121\803121_202...,TIFF missing
9,803121,803121_2025-10-06_17-44-40,iGluSnFR4f\803121\2025-10-06_803121\803121_202...,TIFF missing


## 3. Inspect the exact files

Use this before changing `DRY_RUN`. The full plan can also be exported as a CSV.

In [4]:
cols = [
    "mouse", "session", "relative_path", "tif_exists",
    "dat_exists", "dat_manifested", "eligible", "bytes", "block_reason"
]
display(plan[cols])

LOG_DIR.mkdir(parents=True, exist_ok=True)
stamp = datetime.now().strftime("%Y%m%d_%H%M%S")
plan_path = LOG_DIR / f"slap2_tiff_cleanup_plan_{stamp}.csv"
plan.to_csv(plan_path, index=False)
print(f"Plan saved to:\n{plan_path}")


,mouse,session,relative_path,tif_exists,dat_exists,dat_manifested,eligible,bytes,block_reason
0,803121,803121_2025-10-06_17-44-40,iGluSnFR4f\803121\2025-10-06_803121\803121_202...,False,True,True,False,<NA>,TIFF missing
1,803121,803121_2025-10-06_17-44-40,iGluSnFR4f\803121\2025-10-06_803121\803121_202...,False,True,True,False,<NA>,TIFF missing
2,803121,803121_2025-10-06_17-44-40,iGluSnFR4f\803121\2025-10-06_803121\803121_202...,False,True,True,False,<NA>,TIFF missing
3,803121,803121_2025-10-06_17-44-40,iGluSnFR4f\803121\2025-10-06_803121\803121_202...,False,True,True,False,<NA>,TIFF missing
4,803121,803121_2025-10-06_17-44-40,iGluSnFR4f\803121\2025-10-06_803121\803121_202...,False,True,True,False,<NA>,TIFF missing
...,...,...,...,...,...,...,...,...,...
15936,None,None,test\test_2026-07-23_10-35-41\slap2\dynamic_da...,True,True,True,True,535024016,
15937,None,None,test\test_2026-07-23_10-35-41\slap2\dynamic_da...,True,True,True,True,535024016,
15938,None,None,test\test_2026-07-23_10-35-41\slap2\dynamic_da...,True,True,True,True,535024016,
15939,None,None,test\test_2026-07-23_10-35-41\slap2\dynamic_da...,True,True,True,True,535024016,


Plan saved to:
\\allen\aind\scratch\ophys\Andrew\VIP_synaptic_dynamics\_slap2_cleanup_logs\slap2_tiff_cleanup_plan_20260816_155135.csv


## 4. Dry-run / delete by mouse and session

- With `DRY_RUN=True`, this prints what would happen and deletes nothing.
- With `DRY_RUN=False`, you must set `GLOBAL_CONFIRMATION` exactly to the required phrase.
- If `PROMPT_EACH_SESSION=True`, every session requires an additional typed confirmation.
- Deletion is performed only on rows where `eligible=True`.

In [5]:
def execute_cleanup(plan: pd.DataFrame):
    eligible = plan[plan["eligible"]].copy()

    if eligible.empty:
        print("No eligible TIFFs. Nothing to do.")
        return pd.DataFrame()

    if not DRY_RUN and GLOBAL_CONFIRMATION != REQUIRED_GLOBAL_CONFIRMATION:
        raise RuntimeError(
            "Deletion blocked. Set GLOBAL_CONFIRMATION exactly to:\n"
            f'  "{REQUIRED_GLOBAL_CONFIRMATION}"'
        )

    records = []

    for (mouse, session), group in eligible.groupby(["mouse", "session"], sort=True):
        total_bytes = pd.to_numeric(group["bytes"], errors="coerce").sum(min_count=1)
        mode = "DRY RUN" if DRY_RUN else "DELETE"
        print("\n" + "=" * 80)
        print(f"{mode}: mouse={mouse} | session={session}")
        print(f"{len(group):,} TIFFs | {human_bytes(total_bytes)}")

        session_ok = True
        if not DRY_RUN and PROMPT_EACH_SESSION:
            required = f"DELETE {session}"
            typed = input(f'Type "{required}" to delete this session, or anything else to skip: ')
            session_ok = typed == required
            if not session_ok:
                print("Skipped.")

        for _, row in group.iterrows():
            p = Path(row["tif_path"])
            status = "would_delete" if DRY_RUN else "skipped"
            error = ""

            if not DRY_RUN and session_ok:
                try:
                    # Re-check critical conditions immediately before unlinking.
                    if not p.exists():
                        status = "blocked_missing_tiff"
                    elif REQUIRE_DAT_ON_DISK and not Path(row["dat_path"]).exists():
                        status = "blocked_missing_dat"
                    elif REQUIRE_DAT_IN_MANIFEST and not bool(row["dat_manifested"]):
                        status = "blocked_unmanifested_dat"
                    else:
                        p.unlink()
                        status = "deleted"
                except Exception as exc:
                    status = "error"
                    error = repr(exc)

            records.append({
                "timestamp": datetime.now().isoformat(timespec="seconds"),
                "mouse": mouse,
                "session": session,
                "tif_path": str(p),
                "dat_path": str(row["dat_path"]),
                "bytes": row["bytes"],
                "status": status,
                "error": error,
            })

        session_records = [r for r in records if r["session"] == session and r["mouse"] == mouse]
        counts = pd.Series([r["status"] for r in session_records]).value_counts().to_dict()
        print("Result:", counts)

    result = pd.DataFrame(records)
    stamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    action = "dry_run" if DRY_RUN else "deletion"
    log_path = LOG_DIR / f"slap2_tiff_cleanup_{action}_{stamp}.csv"
    result.to_csv(log_path, index=False)

    print("\n" + "=" * 80)
    print(f"Log saved to:\n{log_path}")
    print(result["status"].value_counts(dropna=False))
    return result

result = execute_cleanup(plan)



DELETE: mouse=826033 | session=826033_2025-12-01_10-44-52
85 TIFFs | 81.13 GB
Result: {'deleted': 85}

DELETE: mouse=826034 | session=826034_2025-12-01_11-32-40
33 TIFFs | 42.84 GB
Result: {'deleted': 33}

DELETE: mouse=834788 | session=834788_2026-03-17_15-17-36
175 TIFFs | 360.42 GB
Result: {'deleted': 175}

DELETE: mouse=834788 | session=834788_2026-03-19_09-05-56
181 TIFFs | 324.29 GB
Result: {'deleted': 181}

DELETE: mouse=834788 | session=834788_2026-03-20_12-44-00
233 TIFFs | 449.64 GB
Result: {'deleted': 233}

DELETE: mouse=836174 | session=836174_2026-06-17_12-51-17
38 TIFFs | 26.10 GB
Result: {'deleted': 38}

DELETE: mouse=838410 | session=838410_2026-03-03_13-49-07
332 TIFFs | 550.20 GB
Result: {'deleted': 332}

DELETE: mouse=838410 | session=838410_2026-03-04_12-54-47
260 TIFFs | 430.29 GB
Result: {'deleted': 260}

DELETE: mouse=838410 | session=838410_2026-03-05_10-16-37
267 TIFFs | 492.25 GB
Result: {'deleted': 267}

DELETE: mouse=838410 | session=838410_2026-03-18_16-43

Result: {'deleted': 166}

DELETE: mouse=863774 | session=863774_2026-08-08_13-35-42
179 TIFFs | 302.59 GB
Result: {'deleted': 179}

DELETE: mouse=863774 | session=863774_2026-08-09_11-37-01
307 TIFFs | 584.38 GB
Result: {'deleted': 307}

DELETE: mouse=875436 | session=875436_2026-07-20_11-20-21
53 TIFFs | 72.01 GB
Result: {'deleted': 53}

Log saved to:
\\allen\aind\scratch\ophys\Andrew\VIP_synaptic_dynamics\_slap2_cleanup_logs\slap2_tiff_cleanup_deletion_20260816_155437.csv
status
deleted    10344
Name: count, dtype: int64


## 5. Post-delete verification

Run after a real deletion. It verifies that targeted TIFFs are gone while their corresponding `.dat` files remain.

In [6]:
def verify_after_cleanup(plan: pd.DataFrame):
    checked = plan[plan["eligible"]].copy()
    checked["tif_exists_now"] = checked["tif_path"].map(lambda p: Path(p).exists())
    checked["dat_exists_now"] = checked["dat_path"].map(lambda p: Path(p).exists())

    report = (
        checked.groupby(["mouse", "session"])
        .agg(
            target_tiffs=("relative_path", "size"),
            tiffs_still_present=("tif_exists_now", "sum"),
            dats_still_present=("dat_exists_now", "sum"),
        )
        .reset_index()
    )
    report["all_target_tiffs_gone"] = report["tiffs_still_present"].eq(0)
    report["all_matching_dats_present"] = report["dats_still_present"].eq(report["target_tiffs"])
    display(report)
    return report

# Uncomment after a real deletion:
verification = verify_after_cleanup(plan)


,mouse,session,target_tiffs,tiffs_still_present,dats_still_present,all_target_tiffs_gone,all_matching_dats_present
0,826033,826033_2025-12-01_10-44-52,85,0,85,True,True
1,826034,826034_2025-12-01_11-32-40,33,0,33,True,True
2,834788,834788_2026-03-17_15-17-36,175,0,175,True,True
3,834788,834788_2026-03-19_09-05-56,181,0,181,True,True
4,834788,834788_2026-03-20_12-44-00,233,0,233,True,True
5,836174,836174_2026-06-17_12-51-17,38,0,38,True,True
6,838410,838410_2026-03-03_13-49-07,332,0,332,True,True
7,838410,838410_2026-03-04_12-54-47,260,0,260,True,True
8,838410,838410_2026-03-05_10-16-37,267,0,267,True,True
9,838410,838410_2026-03-18_16-43-23,217,0,217,True,True
